# Audit checks (read only)

Two checks of the audit document that need to read large files (too slow from the remote session):

1. **BIDS ↔ fMRIPrep correspondence** (point P9): for each run, the fMRIPrep reference image (`desc-hmc_boldref`) is compared with the first volumes of each of the 8 raw BIDS runs of the subject. The best match must be the run with the same name. Important for subjects whose labels changed after fMRIPrep (sub-10, sub-16, sub-01, sub-06).
2. **Which stimulus was shown in each bars run** (points E2, P2, P3): the bars runs are predicted with pRF parameters estimated **without the bars** (RingsWedges fit), using either the bars sequence of the model (version A), the old sequence (version B, from `sourcedata/C/jp/cond-1-stim-3.mat`, built exactly like `create_vdm_files.ipynb`), or version A shifted by ±1-2 TR (trigger delay). The candidate with the highest correlation with the data identifies the stimulus.

Nothing is written in the data folders. Run at the lab (direct disk access); about 1-2 min per subject.

In [1]:
import os, glob, math, gzip
import numpy as np, pandas as pd, nibabel as nb
import scipy.io as sio

main = '/Users/uriel/disks/meso_shared/amblyo7T_prf'
pp = f'{main}/derivatives/pp_data'
subjects = ['sub-%02d' % i for i in [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21]]   # sub-07 = copy of sub-06

/var/folders/60/q71xm4hx45x3db5d6mzl8r4h0000gn/T/ipykernel_1420/2886683577.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import numpy as np, pandas as pd, nibabel as nb


## 1. BIDS ↔ fMRIPrep correspondence

In [2]:
rows = []
for s in subjects + ['sub-07']:
    raws = sorted(glob.glob(f'{main}/{s}/ses-01/func/*_bold.nii.gz'))
    R = {os.path.basename(f).split('_task-')[1].replace('_bold.nii.gz', ''): np.asarray(nb.load(f).dataobj[..., 10:14], dtype=np.float32).mean(3)
         for f in raws}
    for t in R:
        fr = glob.glob(f'{main}/derivatives/fmriprep/fmriprep/{s}/ses-01/func/{s}_ses-01_task-{t}_desc-hmc_boldref.nii.gz')
        if not fr:
            rows.append({'subject': s, 'run': t, 'status': 'no fMRIPrep boldref'}); continue
        b = nb.load(fr[0]).get_fdata(dtype=np.float32)
        cs = {k: np.corrcoef(b.ravel(), v.ravel())[0, 1] for k, v in R.items() if v.shape == b.shape}
        best = max(cs, key=cs.get)
        second = sorted(cs.values())[-2]
        rows.append({'subject': s, 'run': t, 'best_match': best, 'r_own': cs.get(t), 'r_best': cs[best], 'margin': cs[best] - second,
                     'status': 'OK' if best == t else 'MISMATCH'})
    print(s, 'done')
match = pd.DataFrame(rows)
print(match[match.status != 'OK'].to_string() if (match.status != 'OK').any() else 'All fMRIPrep runs match their BIDS run.')
match.groupby('subject').margin.min().round(4)

sub-01 done
sub-02 done
sub-03 done
sub-04 done
sub-05 done
sub-06 done
sub-08 done
sub-09 done
sub-10 done
sub-11 done
sub-12 done
sub-13 done
sub-14 done
sub-16 done
sub-17 done
sub-18 done
sub-19 done
sub-20 done
sub-21 done
sub-07 done
   subject                  run           best_match     r_own    r_best    margin    status
83  sub-12  BarsRightEye_run-02  BarsRightEye_run-01  0.900319  0.925884  0.020163  MISMATCH


subject
sub-01    0.0012
sub-02    0.0020
sub-03    0.0028
sub-04    0.0143
sub-05    0.0022
sub-06    0.0033
sub-07    0.0033
sub-08    0.0055
sub-09    0.0019
sub-10    0.0103
sub-11    0.0010
sub-12    0.0031
sub-13    0.0015
sub-14    0.0023
sub-16    0.0029
sub-17    0.0008
sub-18    0.0043
sub-19    0.0029
sub-20    0.0094
sub-21    0.0016
Name: margin, dtype: float64

## 2. Stimulus of each bars run

Prediction = canonical HRF (SPM double gamma) * (stimulus · Gaussian pRF)^n, with x, y, σ, n of the **RingsWedges** fit of the same eye (independent of the bars). Correlation per vertex (V1-V3, RingsWedges R² > 0.25, eccentricity < 5.5°), median over vertices, for each run of the BarsBars concatenation (run 1 = Bars01, run 2 = Bars02). The design pixel size follows the convention of the current fits (screen height / 50); it is the same for all candidates.

In [3]:
# Design matrices: A = model (derivatives/vdm), B = old bars sequence built like create_vdm_files.ipynb
A = np.load(f'{main}/derivatives/vdm/task-BarsBarsRingsWedgesLeftEye_vdm.npy')[..., :120]

def build_vdm(mat_fn):
    im = np.asarray(sio.loadmat(mat_fn, squeeze_me=True, struct_as_record=False)['s'].im, float)
    out = np.zeros((50, 50, im.shape[-1]))
    idx = np.floor(np.arange(50) * 102 / 50).astype(int)
    for t in range(im.shape[-1]):
        f = (im[..., t] * 255).astype(np.uint8).T
        sq = np.vstack([np.zeros((12, 102), np.uint8), f, np.zeros((13, 102), np.uint8)])
        out[..., t] = (sq > 5)[np.ix_(idx, idx)]
    return out

A_check = build_vdm(f'{main}/sourcedata/C/ac/cond-1-stim-1.mat')
print('rebuilt A vs model A, agreement:', (A_check == A).mean())
B = build_vdm(f'{main}/sourcedata/C/jp/cond-1-stim-3.mat')
print('B vs A correlation:', np.corrcoef(A.ravel(), B.ravel())[0, 1])

pix = 2 * math.degrees(math.atan(50.6 / 2 / 242)) / 50
c_ = (np.arange(50) - 24.5) * pix
X, Y = np.meshgrid(c_, -c_)
t_ = np.arange(0, 32, 2.0)
g = lambda a: t_ ** (a - 1) * np.exp(-t_) / math.gamma(a)
H = g(6) - g(16) / 6; H /= H.sum()


def predict(stim, x0, y0, s, n):
    S = stim.reshape(-1, stim.shape[-1])
    G = np.exp(-((X.ravel()[:, None] - x0) ** 2 + (Y.ravel()[:, None] - y0) ** 2) / (2 * s ** 2))
    d = (S.T @ G) ** n
    return np.apply_along_axis(lambda v: np.convolve(v, H)[:len(v)], 0, d)


def shift(p, k):
    q = np.roll(p, k, 0)
    if k > 0: q[:k] = p[0]
    elif k < 0: q[k:] = p[-1]
    return q


def zc(a):
    a = a - a.mean(0)
    return a / (a.std(0) + 1e-9)

rebuilt A vs model A, agreement: 0.9999133333333333
B vs A correlation: 0.18537931775359762


In [4]:
rows = []
for s in subjects:
    for eye in ['LeftEye', 'RightEye']:
        bold, D, roi = [], [], []
        for h in ['L', 'R']:
            bfn = f'{pp}/{s}/fsnative/func/fmriprep_dct_psc_concat/{s}_task-BarsBars{eye}_hemi-{h}_fmriprep_dct_psc_concat_bold.func.gii'
            dfn = f'{pp}/{s}/fsnative/prfmodel/prf_derivatives/{s}_task-RingsWedges{eye}_hemi-{h}_fmriprep_dct_psc_concat_prf-css_deriv.func.gii'
            rfn = glob.glob(f'{pp}/{s}/fsnative/rois/{s}_hemi-{h}_fmriprep_dct_*_rois-group-mmp.func.gii')
            if not (os.path.isfile(bfn) and os.path.isfile(dfn) and rfn):
                continue
            b = np.vstack([d.data for d in nb.load(bfn).darrays])
            bold.append(b if b.shape[0] == 240 else b.T)
            D.append(np.vstack([d.data for d in nb.load(dfn).darrays]))
            roi.append(nb.load(rfn[0]).darrays[0].data)
        if not bold:
            print(s, eye, 'missing files'); continue
        bold, D, roi = np.hstack(bold), np.hstack(D), np.concatenate(roi)
        ok = (roi >= 1) & (roi <= 3) & (D[0] > 0.25) & (D[1] < 5.5) & np.isfinite(bold).all(0)
        idx = np.where(ok)[0]
        if len(idx) > 3000:
            idx = np.random.default_rng(0).choice(idx, 3000, replace=False)
        x0, y0, sz, n = D[7][idx], D[8][idx], np.clip(D[4][idx], 0.1, None), np.clip(D[11][idx], 0.05, 1)
        pA, pB = predict(A, x0, y0, sz, n), predict(B, x0, y0, sz, n)
        cand = {'A': pA, 'B': pB, 'A-2': shift(pA, -2), 'A-1': shift(pA, -1), 'A+1': shift(pA, 1), 'A+2': shift(pA, 2)}
        for r, run in enumerate(['Bars01', 'Bars02']):
            y = zc(bold[r * 120:(r + 1) * 120, idx])
            res = {k: float(np.median((zc(v) * y).mean(0))) for k, v in cand.items()}
            rows.append({'subject': s, 'eye': eye, 'run': run, 'n_vert': len(idx), **res, 'best': max(res, key=res.get)})
    print(s, 'done')
stim = pd.DataFrame(rows)
stim.round(3)

sub-01 done
sub-02 done
sub-03 done
sub-04 done
sub-05 done
sub-06 done
sub-08 done
sub-09 done
sub-10 done
sub-11 done
sub-12 done
sub-13 done
sub-14 done
sub-16 done
sub-17 done
sub-18 done
sub-19 done
sub-20 done
sub-21 done


,subject,eye,run,n_vert,A,B,A-2,A-1,A+1,A+2,best
0,sub-01,LeftEye,Bars01,3000,0.580,0.048,0.530,0.649,0.356,0.068,A-1
1,sub-01,LeftEye,Bars02,3000,0.552,0.167,0.493,0.606,0.360,0.092,A-1
2,sub-01,RightEye,Bars01,3000,0.533,0.142,0.534,0.621,0.314,0.040,A-1
3,sub-01,RightEye,Bars02,3000,0.577,0.117,0.529,0.631,0.399,0.132,A-1
4,sub-02,LeftEye,Bars01,3000,0.527,0.081,0.579,0.636,0.278,0.007,A-1
...,...,...,...,...,...,...,...,...,...,...,...
71,sub-20,RightEye,Bars02,3000,0.439,0.061,0.457,0.520,0.242,0.009,A-1
72,sub-21,LeftEye,Bars01,3000,0.303,0.003,0.212,0.304,0.189,0.070,A-1
73,sub-21,LeftEye,Bars02,3000,0.300,-0.010,0.290,0.347,0.160,-0.006,A-1
74,sub-21,RightEye,Bars01,3000,0.324,0.034,0.327,0.385,0.176,0.024,A-1


In [5]:
print('Runs whose best candidate is not the model (A):')
print(stim[stim.best != 'A'].round(3).to_string())
print('\nExpected: B for all bars runs of sub-14, sub-16, sub-17; to decide: sub-05 RightEye Bars01; sub-21, sub-18, sub-11, sub-19, sub-12 weak runs.')

Runs whose best candidate is not the model (A):
   subject       eye     run  n_vert      A      B    A-2    A-1    A+1    A+2 best
0   sub-01   LeftEye  Bars01    3000  0.580  0.048  0.530  0.649  0.356  0.068  A-1
1   sub-01   LeftEye  Bars02    3000  0.552  0.167  0.493  0.606  0.360  0.092  A-1
2   sub-01  RightEye  Bars01    3000  0.533  0.142  0.534  0.621  0.314  0.040  A-1
3   sub-01  RightEye  Bars02    3000  0.577  0.117  0.529  0.631  0.399  0.132  A-1
4   sub-02   LeftEye  Bars01    3000  0.527  0.081  0.579  0.636  0.278  0.007  A-1
5   sub-02   LeftEye  Bars02    3000  0.505  0.061  0.537  0.604  0.265  0.006  A-1
6   sub-02  RightEye  Bars01    3000  0.498  0.030  0.597  0.626  0.262  0.013  A-1
7   sub-02  RightEye  Bars02    3000  0.517  0.043  0.580  0.630  0.287  0.037  A-1
8   sub-03   LeftEye  Bars01    3000  0.438  0.029  0.520  0.563  0.214 -0.009  A-1
9   sub-03   LeftEye  Bars02    3000  0.481  0.008  0.479  0.555  0.305  0.094  A-1
10  sub-03  RightEye  Bars01